# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

#from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

random_seed = 1

### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [2]:
# 1.2.3. Load cleaned data as df:

df = pd.read_csv('zillow_cleaned.csv') # using latest csv

In [3]:
# Check df:

df.head()

,baths_calc,baths_full,baths_three_quarter,bedroomcnt,building_quality,fireplaces,garage_spaces,garage_sqft,has_deck,has_hot_tub,...,latitude,longitude,lot_size,sqft,stories,tax_value,total_rooms,units,year_built,zip_code
0,3.5,3.0,1.0,4.0,8.0,0,2.0,633.0,0,0,...,33634931.0,-117869207.0,4506.0,3100.0,1.0,1023282.0,0.0,1.0,1998.0,96978.0
1,1.0,1.0,0.0,2.0,9.0,1,1.0,0.0,0,0,...,34449266.0,-119281531.0,12647.0,1465.0,1.0,464000.0,5.0,1.0,1967.0,97099.0
2,2.0,2.0,0.0,3.0,7.0,0,2.0,440.0,0,0,...,33886168.0,-117823170.0,8432.0,1243.0,1.0,564778.0,6.0,1.0,1962.0,97078.0
3,3.0,3.0,0.0,4.0,8.0,0,0.0,0.0,0,0,...,34245180.0,-118240722.0,13038.0,2376.0,1.0,145143.0,0.0,1.0,1970.0,96330.0
4,3.0,3.0,0.0,3.0,8.0,0,0.0,0.0,0,0,...,34185120.0,-118414640.0,278581.0,1312.0,1.0,119407.0,0.0,1.0,1964.0,96451.0


In [9]:
df.describe()

,baths_calc,baths_full,baths_three_quarter,bedroomcnt,building_quality,fireplaces,garage_spaces,garage_sqft,has_deck,has_hot_tub,...,land_use_type,latitude,longitude,lot_size,sqft,stories,tax_value,total_rooms,units,year_built
count,77377.000000,77377.000000,77377.000000,77377.000000,77377.000000,77377.00000,77377.000000,77377.000000,77377.000000,77377.000000,...,77377.000000,7.737700e+04,7.737700e+04,7.737700e+04,77377.000000,77377.000000,7.737700e+04,77377.000000,77377.000000,77377.000000
mean,2.304199,2.237797,0.131848,3.060832,6.810318,0.12978,0.598666,115.439950,0.007935,0.019890,...,261.822053,3.400868e+07,-1.182039e+08,2.756849e+04,1785.647776,1.098763,4.888552e+05,1.479445,1.072282,1968.598188
std,0.991370,0.978037,0.342881,1.131744,1.564537,0.40390,0.917890,222.846065,0.088726,0.139622,...,5.143680,2.652085e+05,3.591728e+05,1.166800e+05,956.088804,0.317119,6.501854e+05,2.825907,0.948522,23.798031
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,31.000000,3.333953e+07,-1.194754e+08,2.360000e+02,128.000000,1.000000,1.000000e+03,0.000000,1.000000,1824.000000
25%,2.000000,2.000000,0.000000,2.000000,6.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,261.000000,3.381496e+07,-1.184153e+08,5.603000e+03,1182.000000,1.000000,2.069580e+05,0.000000,1.000000,1953.000000
50%,2.000000,2.000000,0.000000,3.000000,7.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,261.000000,3.402244e+07,-1.181810e+08,7.085000e+03,1542.000000,1.000000,3.588780e+05,0.000000,1.000000,1970.000000
75%,3.000000,3.000000,0.000000,4.000000,8.000000,0.00000,2.000000,0.000000,0.000000,0.000000,...,266.000000,3.417451e+07,-1.179292e+08,1.077300e+04,2113.000000,1.000000,5.685400e+05,0.000000,1.000000,1987.000000
max,18.000000,18.000000,7.000000,16.000000,12.000000,5.00000,14.000000,4251.000000,1.000000,1.000000,...,275.000000,3.481877e+07,-1.175546e+08,6.971010e+06,35640.000000,6.000000,4.906124e+07,15.000000,237.000000,2016.000000


In [ ]:


## drop id_city and zip code unless we want to one hot them 

df_first_models = df.drop(columns = ['id_city','zip_code'])



In [ ]:
# 4. Create train/test split:

X = df_first_models.drop(columns=['tax_value'])
y = df_first_models['tax_value']

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state = random_seed)

In [11]:
# 5. Standardize features (not target, not testing data) using standard scalar:

X_train_scaled = StandardScaler().fit_transform(X_train)

# X_train_scaled is now ready to be used in the models
# X_test will have to be scaled later

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [12]:
# Model A: Lasso

from sklearn.linear_model import Lasso
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

lasso = Lasso(random_state=random_seed)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    lasso,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.451e+15, tolerance: 2.247e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.428e+15, tolerance: 2.089e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duali

RMSE scores: [468958.14756455 536466.91824861 478319.70813115 457738.69698942
 643731.25636081 491825.41981028 615108.92233111 536438.0928299
 443858.7097758  503581.11356433 520611.37371004 514604.2973747
 592003.68865178 494224.1846597  477048.44837758 449782.84750904
 595932.6027028  479725.64328196 552518.24249196 560498.93180397
 581394.32707624 514539.12711111 468149.81876272 542373.26309045
 501008.78183334]
Mean RMSE: 520817.7025617342
Std RMSE: 52932.9725453575


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.598e+15, tolerance: 2.276e+12
  model = cd_fast.enet_coordinate_descent(


In [13]:
# Model B: Decision Tree

## Josh model here


from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor
import numpy as np

tree = DecisionTreeRegressor(
    random_state=random_seed
)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    tree,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())



RMSE scores: [534481.7103037  594928.08185689 545680.87931469 581674.22677163
 689133.10061326 658267.45746674 635822.4831754  520560.25143701
 518986.11360714 535221.2772527  578789.27574364 474279.77625244
 659559.81496366 545762.49499846 557589.05579376 515601.617542
 625409.95683088 624896.96595318 584652.8958195  529909.0552109
 595707.65116077 538975.17626222 532927.42338693 593144.35371119
 484780.7240078 ]
Mean RMSE: 570269.6727774593
Std RMSE: 54715.45043245058


In [14]:
# Model C: Gradient Boosting Trees

# Jessica model here

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

gbr = GradientBoostingRegressor(random_state=random_seed)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    gbr,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())



RMSE scores: [409591.14200869 455584.60514359 405908.71597872 408477.54110251
 512464.70313703 432867.93411212 528842.71647884 427387.03652142
 384840.99256967 436797.56334074 453146.65962728 406228.36932696
 494402.62153378 430103.87575804 418142.45488171 400417.02317669
 526955.04828995 422291.6431423  476021.12160883 398400.73937576
 496463.35912637 448177.42597395 412607.98412135 469791.81864855
 375993.97368137]
Mean RMSE: 441276.2827466481
Std RMSE: 42894.07312474434


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [17]:
X_train = X_train.drop(columns=['sqft_log','tax_value_log','lot_size_log'])

In [18]:
X_train.describe()

,baths_calc,baths_full,baths_three_quarter,bedroomcnt,building_quality,fireplaces,garage_spaces,garage_sqft,has_deck,has_hot_tub,...,id_county,land_use_type,latitude,longitude,lot_size,sqft,stories,total_rooms,units,year_built
count,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000,...,61901.000000,61901.000000,6.190100e+04,6.190100e+04,6.190100e+04,61901.000000,61901.000000,61901.000000,61901.000000,61901.000000
mean,2.305278,2.238381,0.132970,3.059870,6.817354,0.130935,0.601977,116.131613,0.007948,0.019870,...,2533.656823,261.832232,3.400874e+07,-1.182040e+08,2.778960e+04,1788.079756,1.099110,1.489265,1.071857,1968.679100
std,0.994014,0.980360,0.344693,1.129843,1.563006,0.406088,0.920468,223.635014,0.088798,0.139556,...,801.347020,5.144818,2.654118e+05,3.597191e+05,1.183137e+05,965.390936,0.317476,2.835678,1.030303,23.787049
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1286.000000,31.000000,3.333953e+07,-1.194754e+08,4.350000e+02,128.000000,1.000000,0.000000,1.000000,1866.000000
25%,2.000000,2.000000,0.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1286.000000,261.000000,3.381401e+07,-1.184161e+08,5.606000e+03,1183.000000,1.000000,0.000000,1.000000,1953.000000
50%,2.000000,2.000000,0.000000,3.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3101.000000,261.000000,3.402276e+07,-1.181810e+08,7.087000e+03,1542.000000,1.000000,0.000000,1.000000,1970.000000
75%,3.000000,3.000000,0.000000,4.000000,8.000000,0.000000,2.000000,0.000000,0.000000,0.000000,...,3101.000000,266.000000,3.417463e+07,-1.179280e+08,1.077300e+04,2114.000000,1.000000,0.000000,1.000000,1987.000000
max,18.000000,18.000000,7.000000,14.000000,12.000000,5.000000,14.000000,4251.000000,1.000000,1.000000,...,3101.000000,275.000000,3.481877e+07,-1.175546e+08,6.971010e+06,35640.000000,6.000000,15.000000,237.000000,2016.000000


In [ ]:
# 3 engineered features saved to df, add back zip code and id_city

X_train['sqft_log'] = np.log(X_train['sqft'])

X_train['lot_size_log'] = np.log(X_train['lot_size'])

X_train = X_train.drop(columns = ['sqft','lot_size'])

y_train = np.log(y_train)



## Zip Code and City Encoding

##combine back
train_with_target = X_train.copy()
train_with_target['tax_value_log'] = y_train

#find the median tax value in each zip code
zip_tax = train_with_target.groupby('zip_code')['tax_value_log'].median()

# rank zip codes into 10 groups based on that median tax value
zip_tax_rank = pd.qcut(zip_tax, q=10, labels=False) + 1

#map that ranking back to each row
X_train['zip_rank_10'] = X_train['zip_code'].map(zip_tax_rank)

## fill any missing zips with middle value
X_train['zip_rank_10'] = X_train['zip_rank_10'].fillna(5)



## City
city_tax = train_with_target.groupby('id_city')['tax_value_log'].median()

city_tax_rank = pd.qcut(city_tax, q=10, labels=False) + 1

X_train['city_rank_10'] = X_train['id_city'].map(city_tax_rank)
X_train['city_rank_10'] = X_train['city_rank_10'].fillna(5)


KeyError: "['tax_value'] not found in axis"

In [ ]:
# Model A using engineered features:

In [ ]:
# Model B using engineered features:

In [ ]:
# Model C using engineered features:

### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Your text here

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# Feature selection for model A:


In [ ]:
# Model A using best features:

In [ ]:
# Model B using best features:

In [ ]:
# Model C using best features:

### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [ ]:
# Hyperparamtere tuning for Model A:


In [ ]:
# Hyperparamtere tuning for Model B:


In [ ]:
# Hyperparamtere tuning for Model C:


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [ ]:
# Rerun of best model with engineered features, best features, and hyperparamter tuning on training data:

In [ ]:
# Model performance on test data:

### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here